# UNet — Conditional Diffusion Training

### Dataset
https://www.kaggle.com/datasets/hypeco/laionart

Trains a conditional UNet on LAION-art using a **custom VAE + custom CLIP encoder** (no diffusers/HuggingFace VAE dependency).

**Bugs fixed vs original:**
- `context_dim=768` in UNet but CLIP outputs `512` — dimension mismatch crash
- Raw timestep integer → `Linear(1,256)` replaced with sinusoidal embedding
- `CrossAttention` missing residual connection + pre-norm (training instability)
- `conv_merge` skip connection was correct structurally but applied before norm — fixed
- Dataset normalised to `[-1,1]` (was `[0.5]` single-channel norm — wrong for 3-ch images)
- Image size was 256×256 but VAE produces 16×16 latents from 128×128 — now consistent
- Removed `diffusers`/HuggingFace VAE dependency — uses the project VAE throughout
- Timestep tensor passed as `.long()` (required by sinusoidal embedding indexing)

In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPModel
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Download LAION-art data

Reads the parquet index and downloads image/caption pairs in parallel.

In [ ]:
df = pd.read_parquet("data/laion-art.parquet")
print(df.shape)
df.head()

In [ ]:
os.makedirs("src/data/laion-art_img", exist_ok=True)
sample_df = df.sample(3000, random_state=42)

def download_image(row_data):
    idx, row = row_data
    try:
        resp = requests.get(row["URL"], timeout=5)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img.save(f"src/data/laion-art_img/{idx}.jpg")
        with open(f"src/data/laion-art_img/{idx}.txt", "w", encoding="utf-8") as f:
            f.write(str(row["TEXT"]))
    except Exception:
        pass  # skip broken URLs silently

rows = list(sample_df.iterrows())
with ThreadPoolExecutor(max_workers=32) as ex:
    list(tqdm(ex.map(download_image, rows), total=len(rows), desc="Downloading"))

downloaded = len([f for f in os.listdir("src/data/laion-art_img") if f.endswith(".jpg")])
print(f"Downloaded: {downloaded} images")

## 2. Models

### 2a. VAE (same as vae.ipynb — used to encode training images)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(8, channels), nn.SiLU(), nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels), nn.SiLU(), nn.Conv2d(channels, channels, 3, padding=1),
        )
    def forward(self, x): return x + self.net(x)


class VAE(nn.Module):
    def __init__(self, in_channels=3, latent_channels=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64,  4, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(64,  128, 4, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.SiLU(),
            ResBlock(256), nn.GroupNorm(8, 256), nn.SiLU(),
            nn.Conv2d(256, latent_channels * 2, 3, padding=1),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_channels, 256, 3, padding=1), nn.SiLU(),
            ResBlock(256),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.SiLU(),
            nn.ConvTranspose2d(128, 64,  4, stride=2, padding=1), nn.SiLU(),
            nn.ConvTranspose2d(64, in_channels, 4, stride=2, padding=1),
            nn.Tanh(),
        )

    def encode(self, x):
        mean, logvar = torch.chunk(self.encoder(x), 2, dim=1)
        logvar = logvar.clamp(-30.0, 20.0)
        z = mean + torch.randn_like(mean) * (0.5 * logvar).exp()
        return z, mean, logvar

    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        z, mean, logvar = self.encode(x)
        return self.decode(z), mean, logvar

### 2b. CLIP Text Encoder

In [ ]:
class TextEncoder(nn.Module):
    """Frozen CLIP + trainable linear projection. Outputs (B, 77, 512)."""
    def __init__(self, model_name="openai/clip-vit-base-patch32", embed_dim=512):
        super().__init__()
        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)
        clip = CLIPModel.from_pretrained(model_name)
        self.transformer = clip.text_model
        self.projection  = nn.Linear(512, embed_dim)
        for p in self.transformer.parameters():
            p.requires_grad = False

    @property
    def device(self): return next(self.parameters()).device

    def forward(self, text_list):
        tokens = self.tokenizer(text_list, padding="max_length", max_length=77,
                                truncation=True, return_tensors="pt")
        tokens = {k: v.to(self.device) for k, v in tokens.items()}
        with torch.no_grad():
            hidden = self.transformer(**tokens).last_hidden_state  # (B, 77, 512)
        return self.projection(hidden)

### 2c. Conditional UNet

In [ ]:
def sinusoidal_embedding(timesteps, dim):
    """
    FIX: original used Linear(1,256) on raw int timesteps.
    Sinusoidal embeddings give scale-invariant, periodic time representation.
    """
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=timesteps.device) / (half - 1))
    args  = timesteps.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class CrossAttention(nn.Module):
    """
    FIX 1: added pre-GroupNorm on spatial features (training stability)
    FIX 2: added residual connection  x = x + attn(x)  (was discarding x entirely)
    """
    def __init__(self, d_model, context_dim, heads=8):
        super().__init__()
        assert d_model % heads == 0
        self.heads    = heads
        self.head_dim = d_model // heads
        self.scale    = self.head_dim ** -0.5
        self.norm   = nn.GroupNorm(8, d_model)
        self.to_q   = nn.Linear(d_model,     d_model, bias=False)
        self.to_k   = nn.Linear(context_dim, d_model, bias=False)
        self.to_v   = nn.Linear(context_dim, d_model, bias=False)
        self.to_out = nn.Linear(d_model, d_model)

    def forward(self, x, context):
        b, c, h, w = x.shape
        x_norm = self.norm(x).view(b, c, h*w).permute(0, 2, 1)
        q = self.to_q(x_norm)
        k = self.to_k(context)
        v = self.to_v(context)
        def split(t): bsz,s,_ = t.shape; return t.view(bsz,s,self.heads,self.head_dim).transpose(1,2)
        q, k, v = split(q), split(k), split(v)
        attn = F.softmax(torch.matmul(q, k.transpose(-2,-1)) * self.scale, dim=-1)
        out  = torch.matmul(attn, v).transpose(1,2).contiguous().view(b, h*w, c)
        out  = self.to_out(out).permute(0,2,1).view(b,c,h,w)
        return x + out  # FIX: residual — was returning out alone


class UNetConditional(nn.Module):
    def __init__(self, latent_channels=4, context_dim=512, time_dim=256):
        # FIX: context_dim=512 (CLIP outputs 512); original notebook had 768 — crash
        super().__init__()
        self.time_dim  = time_dim
        self.time_mlp  = nn.Sequential(nn.Linear(time_dim, time_dim*2), nn.SiLU(), nn.Linear(time_dim*2, time_dim))
        self.time_proj1 = nn.Linear(time_dim, 128)
        self.time_proj2 = nn.Linear(time_dim, 256)

        self.down1 = nn.Conv2d(latent_channels, 128, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, 128)
        self.attn1 = CrossAttention(128, context_dim)

        self.down2 = nn.Conv2d(128, 256, 4, stride=2, padding=1)
        self.norm2 = nn.GroupNorm(8, 256)
        self.attn2 = CrossAttention(256, context_dim)

        self.up1      = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.norm_up  = nn.GroupNorm(8, 128)
        self.out      = nn.Conv2d(128, latent_channels, 3, padding=1)

    def forward(self, x, t, context):
        # FIX: sinusoidal embedding instead of Linear(1,256) on raw int
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))
        t1 = self.time_proj1(t_emb).unsqueeze(-1).unsqueeze(-1)
        t2 = self.time_proj2(t_emb).unsqueeze(-1).unsqueeze(-1)

        h1 = F.silu(self.norm1(self.down1(x) + t1))
        h1 = self.attn1(h1, context)

        h2 = F.silu(self.norm2(self.down2(h1) + t2))
        h2 = self.attn2(h2, context)

        h_up = F.silu(self.norm_up(self.up1(h2) + h1))  # skip + norm
        return self.out(h_up)

## 3. Dataset

In [ ]:
class ImageCaptionDataset(Dataset):
    """
    FIX 1: image_size=128 (was 256 — but VAE expects 128 to produce 16x16 latents)
    FIX 2: Normalize [0.5]*3 on 3 channels (original had [0.5] single value)
    FIX 3: no tokenizer in dataset — text collation happens in train loop
    """
    def __init__(self, image_folder, image_size=128):
        self.image_folder = image_folder
        self.images = [
            f for f in os.listdir(image_folder)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
            and os.path.exists(os.path.join(image_folder, f.rsplit(".", 1)[0] + ".txt"))
        ]
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),  # -> [-1,1]
        ])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]
        img  = self.transform(Image.open(os.path.join(self.image_folder, name)).convert("RGB"))
        with open(os.path.join(self.image_folder, name.rsplit(".",1)[0]+".txt"), encoding="utf-8") as f:
            caption = f.read().strip()
        return img, caption

## 4. Training setup

In [ ]:
dataset    = ImageCaptionDataset("src/data/laion-art_img")
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)
print(f"Dataset: {len(dataset)} pairs, {len(dataloader)} batches")

# Models
vae          = VAE().to(device).eval()
text_encoder = TextEncoder().to(device).eval()
unet         = UNetConditional().to(device)  # context_dim=512 matches CLIP
optimizer    = torch.optim.AdamW(unet.parameters(), lr=1e-4)

# DDPM noise schedule
T      = 1000
betas  = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0).to(device)

# Load VAE if checkpoint exists
vae_ckpt = "models/vae.pth"
if os.path.exists(vae_ckpt):
    vae.load_state_dict(torch.load(vae_ckpt, map_location=device, weights_only=True))
    print(f"Loaded VAE from {vae_ckpt}")
else:
    print("No VAE checkpoint — using random weights (train vae.ipynb first)")

## 5. Shape check

In [ ]:
batch_imgs, batch_caps = next(iter(dataloader))
batch_imgs = batch_imgs.to(device)

with torch.no_grad():
    latents, _, _ = vae.encode(batch_imgs)
    latents_scaled = latents * 0.18215
    context = text_encoder(list(batch_caps))

print(f"Images:  {batch_imgs.shape}         range [{batch_imgs.min():.2f}, {batch_imgs.max():.2f}]")
print(f"Latents: {latents_scaled.shape}    range [{latents_scaled.min():.2f}, {latents_scaled.max():.2f}]")
print(f"Context: {context.shape}")

# Test forward pass
B = batch_imgs.shape[0]
t_test = torch.randint(0, T, (B,), device=device)  # long tensor
noise  = torch.randn_like(latents_scaled)

# Add noise (forward diffusion)
sqrt_acp      = alphas_cumprod[t_test]**0.5
sqrt_one_minus = (1 - alphas_cumprod[t_test])**0.5
while sqrt_acp.dim() < latents_scaled.dim():
    sqrt_acp = sqrt_acp.unsqueeze(-1)
    sqrt_one_minus = sqrt_one_minus.unsqueeze(-1)
noisy   = sqrt_acp * latents_scaled + sqrt_one_minus * noise

# FIX: pass t_test as long (sinusoidal embedding uses it as index)
pred = unet(noisy, t_test, context)
print(f"Noise pred: {pred.shape}  (should match latents shape)")
assert pred.shape == latents_scaled.shape
print("All shapes OK")

## 6. Training loop

In [ ]:
EPOCHS = 10
os.makedirs("models", exist_ok=True)

def add_noise(x0, noise, timesteps):
    """Inline DDPM forward process."""
    a = alphas_cumprod[timesteps]**0.5
    b = (1 - alphas_cumprod[timesteps])**0.5
    while a.dim() < x0.dim(): a = a.unsqueeze(-1); b = b.unsqueeze(-1)
    return a * x0 + b * noise

for epoch in range(EPOCHS):
    unet.train()
    total_loss = 0.0

    bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, captions in bar:
        images = images.to(device)

        with torch.no_grad():
            latents, _, _ = vae.encode(images)
            latents = latents * 0.18215
            context = text_encoder(list(captions))

        noise     = torch.randn_like(latents)
        # FIX: keep timesteps as long for sinusoidal embedding indexing
        timesteps = torch.randint(0, T, (latents.shape[0],), device=device)
        noisy     = add_noise(latents, noise, timesteps)

        noise_pred = unet(noisy, timesteps, context)
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch {epoch+1}  avg_loss={total_loss/len(dataloader):.4f}")

torch.save(unet.state_dict(), "models/unet_final.pth")
print("Saved -> models/unet_final.pth")